# 01. Data Acquisition and Initial Audit

This notebook loads a reproducible study sample from the Open Australian Legal
Corpus and performs an initial inspection before any cleaning or modeling.

Dataset source:
https://huggingface.co/datasets/isaacus/open-australian-legal-corpus

The complete corpus is large, so this project uses a reproducible sample of
1,000 legal documents. The sample is selected after buffered random shuffling
with a fixed random seed.

##1. Environment Setup

In [ ]:
!pip install -q datasets pandas matplotlib

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

##2. Sampling Configuration

In [ ]:
SEED = 42
SAMPLE_SIZE = 1000
SHUFFLE_BUFFER = 20_000

### Sampling decision

A sample of 1,000 documents is used to keep the later retrieval experiments
computationally practical while still covering a broad range of Australian
legal material.

A fixed random seed is used for reproducibility. Because the dataset is loaded
in streaming mode, shuffling is performed through a 20,000-record buffer rather
than loading the complete corpus into memory.

##3. Data Acquisition and Sampling

In [ ]:
dataset = load_dataset(
    "isaacus/open-australian-legal-corpus",
    split="corpus",
    streaming=True
)
dataset


README.md:   0%|          | 0.00/15.7k [00:00<?, ?B/s]

IterableDataset({
    features: ['version_id', 'type', 'jurisdiction', 'source', 'mime', 'date', 'citation', 'url', 'when_scraped', 'text'],
    num_shards: 1
})

In [ ]:
shuffled_dataset = dataset.shuffle(
    seed=SEED,
    buffer_size=SHUFFLE_BUFFER
)
sample = list(
    shuffled_dataset.take(SAMPLE_SIZE)
)
print("Number of sampled documents:", len(sample))

Number of sampled documents: 1000


##4. Dataset Creation and Preview

In [ ]:
df = pd.DataFrame(sample)
print("Object type:", type(df))
print("Dataset shape:", df.shape)

Object type: <class 'pandas.core.frame.DataFrame'>
Dataset shape: (1000, 10)


In [ ]:
print("Columns:")
for column in df.columns:
    print("-", column)
display(
    df[["version_id","type","jurisdiction","source","date","citation"]].head()
)
print(df.loc[0, "text"][:1000])


Columns:
- version_id
- type
- jurisdiction
- source
- mime
- date
- citation
- url
- when_scraped
- text


,version_id,type,jurisdiction,source,date,citation
0,tasmanian_legislation:2000-08-02/sr-2000-118,secondary_legislation,tasmania,tasmanian_legislation,2000-08-02 00:00:00,Proclamation under the National Parks and Wild...
1,nsw_caselaw:5ae13d1de4b074a7c6e1e9a9,decision,new_south_wales,nsw_caselaw,2018-04-27 00:00:00,R v Oncu; R v Dogan; R v Unver; R v Bassi-Anst...
2,nsw_caselaw:59000bb2e4b0e71e17f5902d,decision,new_south_wales,nsw_caselaw,2017-04-27 00:00:00,CPU v Children’s Guardian [2017] NSWCATAD 131
3,tasmanian_legislation:2020-12-02/sr-2020-081,secondary_legislation,tasmania,tasmanian_legislation,2020-12-02 00:00:00,Fisheries (Processing and Handling) Amendment ...
4,tasmanian_legislation:2000-12-27/sr-2000-244,secondary_legislation,tasmania,tasmanian_legislation,2000-12-27 00:00:00,Proclamation under the National Parks and Wild...


Proclamation under the National Parks and Wildlife Act 1970

I, the Lieutenant-Governor in and over the State of Tasmania and its Dependencies in the Commonwealth of Australia, acting with the advice of the Executive Council, by this my proclamation made under section 15C(1) of the National Parks and Wildlife Act 1970 –

        (a) give the area of land specified in Schedule 1 to this proclamation, declared to be a sanctuary by Statutory Rules 1968, No.128 and deemed to have become reserved land in the class of conservation area under Schedule 2 to the National Parks and Wildlife Act 1970 and subsequently extended by the proclamations notified in the Gazette on 24 May 1972 at page number 757 and 7 September 1977 at page number 2276, the name Logan Lagoon Conservation Area; and
        (b) declare that this proclamation takes effect on the day on which its making is notified in the Gazette.
26 July 2000

W. J. E. COX

Lieutenant-Governor

By His Excellency's Command,

D. E. LLEWELLYN



The complete legal text can be very long, so only the first 1,000 characters of
one document are displayed here for inspection.

##5. Initial Data Audit

In [ ]:
df.info()
display(
    df.describe(include="all").T
)
display(
    df["type"]
    .value_counts()
    .rename("count")
    .to_frame()
)
type_distribution = (
    df["type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
display(
    type_distribution
    .rename("percentage")
    .to_frame()
)
display(
    df["jurisdiction"]
    .value_counts()
    .rename("count")
    .to_frame()
)

display(
    df["source"]
    .value_counts()
    .rename("count")
    .to_frame()
)
display(
    pd.crosstab(
        df["jurisdiction"],
        df["type"]
    )
)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   version_id    1000 non-null   object
 1   type          1000 non-null   object
 2   jurisdiction  1000 non-null   object
 3   source        1000 non-null   object
 4   mime          1000 non-null   object
 5   date          893 non-null    object
 6   citation      1000 non-null   object
 7   url           1000 non-null   object
 8   when_scraped  1000 non-null   object
 9   text          1000 non-null   object
dtypes: object(10)
memory usage: 78.3+ KB


,count,unique,top,freq
version_id,1000,1000,tasmanian_legislation:2000-08-02/sr-2000-118,1
type,1000,4,decision,616
jurisdiction,1000,6,new_south_wales,367
source,1000,8,nsw_caselaw,283
mime,1000,4,text/html,855
date,893,759,2024-07-01 00:00:00,13
citation,1000,995,Proclamation under the Nature Conservation Act...,4
url,1000,1000,https://www.legislation.tas.gov.au/view/whole/...,1
when_scraped,1000,1000,2024-09-13T22:47:55.638633+10:00,1
text,1000,1000,Proclamation under the National Parks and Wild...,1


,count
type,
decision,616
secondary_legislation,180
primary_legislation,169
bill,35


,percentage
type,
decision,61.6
secondary_legislation,18.0
primary_legislation,16.9
bill,3.5


,count
jurisdiction,
new_south_wales,367
commonwealth,333
tasmania,114
western_australia,74
queensland,61
south_australia,51


,count
source,
nsw_caselaw,283
federal_court_of_australia,278
tasmanian_legislation,114
nsw_legislation,84
western_australian_legislation,74
queensland_legislation,61
high_court_of_australia,55
south_australian_legislation,51


type,bill,decision,primary_legislation,secondary_legislation
jurisdiction,,,,
commonwealth,0,333,0,0
new_south_wales,0,283,57,27
queensland,22,0,19,20
south_australia,13,0,19,19
tasmania,0,0,34,80
western_australia,0,0,40,34


##6. Saving the Raw Dataset and Metadata

In [ ]:
RAW_DATA_DIR = Path("data/raw")
RAW_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)
RAW_SAMPLE_PATH = (
    RAW_DATA_DIR /
    "australian_legal_corpus_sample.csv.gz"
)
df.to_csv(
    RAW_SAMPLE_PATH,
    index=False,
    compression="gzip"
)
print("Saved raw sample to:")
print(RAW_SAMPLE_PATH)

Saved raw sample to:
data/raw/australian_legal_corpus_sample.csv.gz


In [ ]:
sampling_info = {
    "dataset": "isaacus/open-australian-legal-corpus",
    "dataset_url": (
        "https://huggingface.co/datasets/"
        "isaacus/open-australian-legal-corpus"
    ),
    "split": "corpus",
    "sampling_method": "streaming buffered shuffle",
    "sample_size": SAMPLE_SIZE,
    "random_seed": SEED,
    "shuffle_buffer": SHUFFLE_BUFFER
}
metadata_path = (
    RAW_DATA_DIR /
    "sampling_metadata.json"
)
with open(metadata_path, "w") as f:
    json.dump(
        sampling_info,
        f,
        indent=4
    )
print("Saved metadata to:")
print(metadata_path)

Saved metadata to:
data/raw/sampling_metadata.json


## 7. Initial audit summary

A reproducible sample of 1,000 documents was selected from the Open Australian
Legal Corpus using streaming buffered shuffling with a fixed random seed. The sample contains multiple legal document types, jurisdictions and source
collections. The initial inspection also shows that some fields may contain
missing values and that the category frequencies are not necessarily balanced. No data has been removed or transformed at this stage. These issues will be
investigated systematically in the data-cleaning notebook.